# 🔵 Interagindo com Cassandra usando Python

Este notebook é um guia didático e prático que demonstra como interagir com o **Apache Cassandra** (banco de dados NoSQL do tipo **Família de Colunas** ou **Wide Column Store**) utilizando a linguagem Python e a biblioteca `cassandra-driver`.

## 🛠️ O que é o Cassandra?
Desenvolvido originalmente pelo Facebook, o Apache Cassandra foi projetado para gerenciar grandes volumes de dados distribuídos em vários servidores. Ele oferece alta disponibilidade sem pontos únicos de falha. Ao contrário de bancos de dados relacionais e do MongoDB, no Cassandra, a modelagem de dados deve ser orientada estritamente pelas **consultas** que sua aplicação fará (Query-Driven Modeling).

### Detalhes da Conexão Local (Docker Compose):
- **Host:** `localhost`
- **Porta:** `9042`
- **Autenticação:** Nenhuma (configuração padrão local)

> **Importante:** O Cassandra pode demorar entre 1 e 2 minutos para inicializar completamente no Docker. Certifique-se de que ele já esteja ativo antes de executar o notebook.

## 1. Instalação do Cliente Python
Para nos conectarmos ao Cassandra, instalaremos o driver oficial do Python desenvolvido pela DataStax.

In [ ]:
!pip install cassandra-driver

## 2. Conectando ao Cluster
No Cassandra, conectamo-nos a um cluster composto de um ou mais nós. O driver gerencia a descoberta e balanceamento de conexões automaticamente.

In [ ]:
from cassandra.cluster import Cluster

try:
    # Definir os nós de contato (contact points) e a porta
    cluster = Cluster(['localhost'], port=9042)
    
    # Estabelecer a sessão de comunicação
    session = cluster.connect()
    
    # Executar uma query simples de sistema para verificar a conexão
    resultado = session.execute("SELECT cluster_name, release_version FROM system.local")
    for linha in resultado:
        print(f"✅ Conectado ao cluster: '{linha.cluster_name}' | Versão do Cassandra: {linha.release_version}")
        
except Exception as e:
    print(f"❌ Erro ao conectar ao Cassandra: {e}")
    print("Dica: Aguarde mais um pouco se o container do Cassandra tiver acabado de iniciar.")

## 3. Criando um Keyspace
No Cassandra, um **Keyspace** é equivalente a um banco de dados nos sistemas tradicionais. Ele define o escopo físico da replicação dos dados.

Utilizaremos o fator de replicação `1` com a estratégia simples (`SimpleStrategy`), pois estamos rodando um cluster de desenvolvimento com um único nó local.

In [ ]:
# Criar Keyspace se não existir
query_keyspace = """
CREATE KEYSPACE IF NOT EXISTS escola 
WITH replication = {
    'class': 'SimpleStrategy', 
    'replication_factor': 1
};
"""
session.execute(query_keyspace)
print("🏢 Keyspace 'escola' criado ou já existente.")

# Mudar para o contexto do keyspace criado
session.set_keyspace('escola')
print("🎯 Sessão apontando para o Keyspace 'escola'.")

## 4. Criando Tabelas e a Estrutura da Chave Primária
No Cassandra, a chave primária (`PRIMARY KEY`) é dividida em duas partes muito importantes:
1. **Partition Key (Chave de Partição):** Define em qual nó do cluster o dado físico será gravado. O Cassandra faz o hash desse campo para distribuir os registros uniformemente.
2. **Clustering Key (Chave de Agrupamento):** Define a ordenação física dos dados dentro do nó de partição.

Nesse exemplo, criaremos uma tabela `estudantes` onde a Primary Key é `((curso), id)`. 
- `curso` é a **Partition Key** (todos os alunos de um mesmo curso ficarão salvos juntos fisicamente).
- `id` é a **Clustering Key** (dentro do curso, os alunos serão ordenados pelo ID).

In [ ]:
# Remover tabela se existir para resetar os testes
session.execute("DROP TABLE IF EXISTS estudantes")

# Criar Tabela
query_tabela = """
CREATE TABLE estudantes (
    curso text,
    id int,
    nome text,
    email text,
    nota float,
    PRIMARY KEY ((curso), id)
);
"""
session.execute(query_tabela)
print("📋 Tabela 'estudantes' criada com sucesso!")

## 5. CRUD - Create (Inserir Dados com Prepared Statements)
No Cassandra, é uma excelente prática utilizar **Prepared Statements** (consultas preparadas). Elas reduzem o overhead de parsing de SQL/CQL no banco e previnem injeção de código.

In [ ]:
# Preparar a query de inserção
query_inserir = "INSERT INTO estudantes (curso, id, nome, email, nota) VALUES (?, ?, ?, ?, ?)"
statement_preparado = session.prepare(query_inserir)

# Dados dos estudantes
estudantes = [
    ('Ciência da Computação', 1, 'Felipe Souza', 'felipe@email.com', 8.5),
    ('Ciência da Computação', 2, 'Ana Costa', 'ana.costa@email.com', 9.8),
    ('Sistemas de Informação', 3, 'Carlos Lima', 'carlos@email.com', 7.2),
    ('Sistemas de Informação', 4, 'Beatriz Santos', 'beatriz@email.com', 9.0)
]

# Executar inserção em lote sequencial
for est in estudantes:
    session.execute(statement_preparado, est)
    print(f"✍️ Estudante '{est[2]}' inserido no curso '{est[0]}'.")

## 6. CRUD - Read (Coletar e Consultar Dados)
### Regra de Ouro do Cassandra:
Você **só deve** consultar dados passando a **Partition Key** na cláusula `WHERE` (no nosso caso, o `curso`). 
Tentar buscar dados por colunas não indexadas (como `nome`) causará um erro, a menos que você force a varredura total de todos os nós do banco com `ALLOW FILTERING` (o que é desencorajado em produção porque arruína a performance).

In [ ]:
# === Consulta Eficiente (Filtrando pela Partition Key: curso) ===
print("📖 Consultando alunos de 'Ciência da Computação':")
resultados_cc = session.execute("SELECT id, nome, email, nota FROM estudantes WHERE curso = 'Ciência da Computação'")
for linha in resultados_cc:
    print(f"- ID: {linha.id} | Nome: {linha.nome} | Nota: {linha.nota}")

print("-" * 50)

# === Consulta Completa (Válida, mas sem WHERE) ===
print("📖 Buscando todos os estudantes cadastrados (Select *):")
todos = session.execute("SELECT * FROM estudantes")
for estudante in todos:
    print(f"Curso: {estudante.curso} | ID: {estudante.id} | Nome: {estudante.nome} | Email: {estudante.email}")

print("-" * 50)

# === Consulta Ineficiente / Bloqueada sem index (Tentando filtrar por e-mail) ===
try:
    print("Tentando buscar por email...")
    session.execute("SELECT * FROM estudantes WHERE email = 'felipe@email.com'")
except Exception as e:
    print(f"⚠️ Erro esperado capturado: {e}")
    print("O Cassandra impede essa busca porque 'email' não faz parte da chave primária e não é indexado.")
    
    print("\n🔄 Executando com 'ALLOW FILTERING' (permitindo varredura de tabelas):")
    resultados_filtro = session.execute("SELECT * FROM estudantes WHERE email = 'felipe@email.com' ALLOW FILTERING")
    for linha in resultados_filtro:
        print(f"- Recuperado com sucesso: {linha.nome} ({linha.curso})")

## 7. CRUD - Update (Atualizar Dados)
No Cassandra, a escrita funciona como um **Upsert** (Update + Insert). Se você rodar um `UPDATE` ou `INSERT` passando a mesma chave primária composta, o Cassandra simplesmente sobrescreve os dados existentes.

In [ ]:
# === Atualizar a nota e e-mail do Felipe (Curso: Ciência da Computação, ID: 1) ===
# A chave primária COMPLETA deve ser especificada no WHERE
query_update = """
UPDATE estudantes 
SET nota = 9.5, email = 'felipe.novo@email.com' 
WHERE curso = 'Ciência da Computação' AND id = 1
"""
session.execute(query_update)
print("🔄 Registro de Felipe Souza atualizado!")

# Validar atualização
registro_atualizado = session.execute(
    "SELECT * FROM estudantes WHERE curso = 'Ciência da Computação' AND id = 1"
).one()
print(f"👤 Dados atuais: Nome: {registro_atualizado.nome} | Email: {registro_atualizado.email} | Nota: {registro_atualizado.nota}")

## 8. CRUD - Delete (Deletar Registros)
Para deletar, também precisamos informar a chave de partição (e preferencialmente a chave de agrupamento) para localizar a linha exata.

In [ ]:
# === Deletar estudante Carlos Lima (Sistemas de Informação, ID: 3) ===
query_delete = "DELETE FROM estudantes WHERE curso = 'Sistemas de Informação' AND id = 3"
session.execute(query_delete)
print("🗑️ Carlos Lima foi removido.")

# Exibir lista final de Sistemas de Informação
print("📖 Alunos de Sistemas de Informação restantes:")
restantes_si = session.execute("SELECT * FROM estudantes WHERE curso = 'Sistemas de Informação'")
for est in restantes_si:
    print(f"- ID: {est.id} | Nome: {est.nome}")

## 🏁 Conclusão
Parabéns! Você concluiu os testes práticos com o Apache Cassandra:
- Criou um Keyspace e definiu fatores de replicação locais.
- Modelou uma tabela estruturada sob uma Primary Key Composta (Partition Key + Clustering Key).
- Entendeu a distribuição física dos dados com base na Partition Key.
- Utilizou Prepared Statements para inserção segura.
- Entendeu as restrições de filtragem e como o Cassandra prioriza consultas eficientes através de chaves, evitando varreduras completas desnecessárias.